In [1]:
import os

In [2]:
import cudf
import cupy
from tqdm import tqdm
import numpy as np
import gc
import xgboost as xgb
from utils import amex_metric_np
from pathlib import Path

cudf.__version__, xgb.__version__

('25.10.00', '3.0.5')

# Please register kaggle and install kaggle API by: 
- `pip install kaggle`
- complete [authentication](https://www.kaggle.com/docs/api)

In [3]:
PATH = 'data/amex'

In [4]:
Path(PATH).mkdir(parents=True,exist_ok=True)

In [5]:
cmd = f'kaggle datasets download -d raddar/amex-data-integer-dtypes-parquet-format -p {PATH}'

In [6]:
os.system(cmd)

Dataset URL: https://www.kaggle.com/datasets/raddar/amex-data-integer-dtypes-parquet-format
License(s): unknown


 82%|████████▏ | 3.33G/4.07G [00:00<00:00, 3.89GB/s]

100%|██████████| 4.07G/4.07G [00:01<00:00, 3.94GB/s]


0

In [7]:
os.listdir(PATH)

['amex-data-integer-dtypes-parquet-format.zip']

In [8]:
cmd = f'cd {PATH} && unzip amex-data-integer-dtypes-parquet-format.zip'
os.system(cmd)

Archive:  amex-data-integer-dtypes-parquet-format.zip
  inflating: test.parquet            
  inflating: train.parquet           


0

In [9]:
os.listdir(PATH)

['train.parquet',
 'amex-data-integer-dtypes-parquet-format.zip',
 'test.parquet']

# Basic EDA

In [10]:
%%time
train = cudf.read_parquet(f'{PATH}/train.parquet')
print(train.shape)
train.head()

(5531451, 190)
CPU times: user 471 ms, sys: 832 ms, total: 1.3 s
Wall time: 1.04 s


,customer_ID,S_2,P_2,D_39,B_1,B_2,R_1,S_3,D_41,B_3,...,D_136,D_137,D_138,D_139,D_140,D_141,D_142,D_143,D_144,D_145
0,0000099d6bd597052cdcda90ffabf56573fe9d7c79be5f...,2017-03-09,0.938469,0,0.008724,1.006838,0.009228,0.124035,0.0,0.004709,...,-1,-1,-1,0,0,0.0,<NA>,0,0.000610,0
1,0000099d6bd597052cdcda90ffabf56573fe9d7c79be5f...,2017-04-07,0.936665,0,0.004923,1.000653,0.006151,0.126750,0.0,0.002714,...,-1,-1,-1,0,0,0.0,<NA>,0,0.005492,0
2,0000099d6bd597052cdcda90ffabf56573fe9d7c79be5f...,2017-05-28,0.954180,3,0.021655,1.009672,0.006815,0.123977,0.0,0.009423,...,-1,-1,-1,0,0,0.0,<NA>,0,0.006986,0
3,0000099d6bd597052cdcda90ffabf56573fe9d7c79be5f...,2017-06-13,0.960384,0,0.013683,1.002700,0.001373,0.117169,0.0,0.005531,...,-1,-1,-1,0,0,0.0,<NA>,0,0.006527,0
4,0000099d6bd597052cdcda90ffabf56573fe9d7c79be5f...,2017-07-16,0.947248,0,0.015193,1.000727,0.007605,0.117325,0.0,0.009312,...,-1,-1,-1,0,0,0.0,<NA>,0,0.008126,0


In [11]:
%%time
count_df = train.groupby('customer_ID').size().to_frame('num_profiles')
count_df.head()

CPU times: user 10.4 ms, sys: 8.88 ms, total: 19.3 ms
Wall time: 26.6 ms


,num_profiles
customer_ID,
c761f5f5b15e563daa67f0a41c3ec2a870d3c9daaadf0cd11dd808d3aaa82c46,13
e16b5594d9dce9ebd2f8e0d7074391736b2641afa9e349f67a53f7cc780c120b,13
8c846c26e1f1d4afa04977155c41bc3b6bb77c72efc5db3f592ec3d72f12cfdc,13
463e8a9b5b0161764bbbb0b5b58956bb8ebff6244219b21ac257a07364fa8dd9,13
92bbe3e2a159bcc838b86241471eb14153c8d712b6647feffbe49d5266cdfd3f,13


In [12]:
count_df.num_profiles.max()

np.int64(13)

In [13]:
%%time
train['S_2'] = cudf.to_datetime(train['S_2'])
train.head()

CPU times: user 15.3 ms, sys: 16.9 ms, total: 32.3 ms
Wall time: 45.6 ms


,customer_ID,S_2,P_2,D_39,B_1,B_2,R_1,S_3,D_41,B_3,...,D_136,D_137,D_138,D_139,D_140,D_141,D_142,D_143,D_144,D_145
0,0000099d6bd597052cdcda90ffabf56573fe9d7c79be5f...,2017-03-09,0.938469,0,0.008724,1.006838,0.009228,0.124035,0.0,0.004709,...,-1,-1,-1,0,0,0.0,<NA>,0,0.000610,0
1,0000099d6bd597052cdcda90ffabf56573fe9d7c79be5f...,2017-04-07,0.936665,0,0.004923,1.000653,0.006151,0.126750,0.0,0.002714,...,-1,-1,-1,0,0,0.0,<NA>,0,0.005492,0
2,0000099d6bd597052cdcda90ffabf56573fe9d7c79be5f...,2017-05-28,0.954180,3,0.021655,1.009672,0.006815,0.123977,0.0,0.009423,...,-1,-1,-1,0,0,0.0,<NA>,0,0.006986,0
3,0000099d6bd597052cdcda90ffabf56573fe9d7c79be5f...,2017-06-13,0.960384,0,0.013683,1.002700,0.001373,0.117169,0.0,0.005531,...,-1,-1,-1,0,0,0.0,<NA>,0,0.006527,0
4,0000099d6bd597052cdcda90ffabf56573fe9d7c79be5f...,2017-07-16,0.947248,0,0.015193,1.000727,0.007605,0.117325,0.0,0.009312,...,-1,-1,-1,0,0,0.0,<NA>,0,0.008126,0


In [14]:
train.S_2.min(), train.S_2.max()

(np.datetime64('2017-03-01T00:00:00.000000000'),
 np.datetime64('2018-03-31T00:00:00.000000000'))

## Download the training data labels

In [15]:
cmd = f'kaggle competitions download -c amex-default-prediction -f train_labels.csv -p {PATH}/'
os.system(cmd)

100%|██████████| 16.2M/16.2M [00:00<00:00, 3.56GB/s]


0

In [16]:
cmd = f'cd {PATH} && mv train_labels.csv train_labels.zip && unzip train_labels.zip'
os.system(cmd)

Archive:  train_labels.zip
  inflating: train_labels.csv        


0

In [17]:
%%time
trainl = cudf.read_csv(f'{PATH}/train_labels.csv')
print(trainl.shape)
trainl.head()

(458913, 2)
CPU times: user 9.27 ms, sys: 16.2 ms, total: 25.4 ms
Wall time: 28.5 ms


,customer_ID,target
0,0000099d6bd597052cdcda90ffabf56573fe9d7c79be5f...,0
1,00000fd6641609c6ece5454664794f0340ad84dddce9a2...,0
2,00001b22f846c82c51f6e3958ccd81970162bae8b007e8...,0
3,000041bdba6ecadd89a52d11886e8eaaec9325906c9723...,0
4,00007889e4fcd2614b6cbe7f8f3d2e5c728eca32d9eb8a...,0


In [18]:
trainl['target'].value_counts()

target
0    340085
1    118828
Name: count, dtype: int64

In [19]:
%%time
train = train.merge(trainl, on='customer_ID', how='left')
print(train.shape)
train.head()

(5531451, 191)
CPU times: user 66.1 ms, sys: 7.7 ms, total: 73.8 ms
Wall time: 75.9 ms


,customer_ID,S_2,P_2,D_39,B_1,B_2,R_1,S_3,D_41,B_3,...,D_137,D_138,D_139,D_140,D_141,D_142,D_143,D_144,D_145,target
0,00e2409ac675768f14a25846f8104762d02bc126eae6b2...,2017-04-28,0.345678,0,0.073722,0.817443,0.000588,0.161948,0.000000,0.094881,...,-1,-1,0,0,0.0,<NA>,0,0.005960,0,0
1,00e2409ac675768f14a25846f8104762d02bc126eae6b2...,2017-05-31,0.421038,21,0.141693,0.817831,0.006131,0.172564,0.000000,0.085875,...,-1,-1,0,0,0.0,<NA>,0,0.007887,0,0
2,00e2409ac675768f14a25846f8104762d02bc126eae6b2...,2017-06-09,0.438867,7,0.145901,0.814675,0.004407,0.166595,0.110788,0.052232,...,-1,-1,0,0,0.0,<NA>,0,0.009526,0,0
3,00e2409ac675768f14a25846f8104762d02bc126eae6b2...,2017-07-21,0.423394,18,0.105497,1.007269,0.008933,0.156157,0.000000,0.035599,...,-1,-1,0,0,0.0,<NA>,0,0.002913,0,0
4,00e2409ac675768f14a25846f8104762d02bc126eae6b2...,2017-08-29,0.417205,19,0.098453,1.006110,0.000935,0.153031,0.000000,0.053045,...,-1,-1,0,0,0.0,<NA>,0,0.000781,0,0


In [20]:
train['cid'], _ = train.customer_ID.factorize()

In [21]:
mask = train['cid']%4 == 0
tr,va = train.loc[~mask],train.loc[mask]
print("Verify target distribution is consistent across tr and va")
print(tr['target'].mean(), va['target'].mean())

Verify target distribution is consistent across tr and va
0.24958439479276864 0.24763539266440152
